# Exercise Validation System

## Real-time Exercise Validation using YOLOv8 Pose Estimation

This notebook contains a complete system for:
- **Pose Detection**: Using YOLOv8 pose model
- **Exercise Validation**: Joint angle analysis and repetition counting
- **Error Correction**: Real-time feedback on form
- **Voice Feedback**: Audio cues for exercise guidance

### Supported Exercises:
1. Bending knee no support seated
2. Bending knee with bed support supine
3. Bending knee with support seated
4. Circular pendulum standing
5. External rotation shoulders elastic
6. Horizontal weighted openings standing
7. Lift extended leg supine
8. Shoulder flexion seated

## 1. Install Requirements

Run this cell to install the required packages.

In [ ]:
# Install required packages
!pip install ultralytics>=8.0.0 opencv-python>=4.8.0 numpy>=1.24.0 pyttsx3

## 2. Import Dependencies

In [ ]:
import cv2
import numpy as np
import time
import threading
import pyttsx3
from ultralytics import YOLO
from dataclasses import dataclass
from typing import Tuple, Dict, Optional, List
from enum import Enum
from collections import deque

## 3. Exercise Rules Definition

This section defines the error detection dataclass and rules for all 8 exercises.

In [ ]:
@dataclass
class ErrorDetection:
    error_type: str
    severity: str
    recommendation: str
    joint_name: str
    current_value: float
    expected_range: Tuple[float, float]


EXERCISE_RULES = {
    'bending_knee_no_support_seated': {
        'joint': 'knee',
        'min_angle': 70,
        'max_angle': 180,
        'errors': {
            'insufficient_flexion': {
                'condition': lambda angle, min_a, max_a: angle > min_a + 20,
                'recommendation': "Bend your knee more",
                'severity': 'medium'
            },
            'excessive_flexion': {
                'condition': lambda angle, min_a, max_a: angle < min_a - 10,
                'recommendation': "Do not over-bend your knee",
                'severity': 'high'
            },
            'insufficient_extension': {
                'condition': lambda angle, min_a, max_a: angle < max_a - 20,
                'recommendation': "Straighten your leg fully",
                'severity': 'medium'
            },
            'excessive_extension': {
                'condition': lambda angle, min_a, max_a: angle > max_a + 5,
                'recommendation': "Relax your knee slightly",
                'severity': 'low'
            }
        },
        'movement_speed_check': True,
        'min_rep_duration': 2.0
    },

    'bending_knee_bed_support_supine': {
        'joint': 'knee',
        'min_angle': 60,
        'max_angle': 150,
        'errors': {
            'insufficient_flexion': {
                'condition': lambda angle, min_a, max_a: angle > min_a + 15,
                'recommendation': "Bend your knee closer to your chest",
                'severity': 'medium'
            },
            'excessive_flexion': {
                'condition': lambda angle, min_a, max_a: angle < min_a - 10,
                'recommendation': "Do not pull your knee too close",
                'severity': 'high'
            },
            'insufficient_extension': {
                'condition': lambda angle, min_a, max_a: angle < max_a - 15,
                'recommendation': "Extend your leg more",
                'severity': 'medium'
            }
        },
        'movement_speed_check': True,
        'min_rep_duration': 2.5
    },

    'bending_knee_with_support_seated': {
        'joint': 'knee',
        'min_angle': 80,
        'max_angle': 160,
        'errors': {
            'insufficient_flexion': {
                'condition': lambda angle, min_a, max_a: angle > min_a + 15,
                'recommendation': "Bend your knee a bit more",
                'severity': 'medium'
            },
            'insufficient_extension': {
                'condition': lambda angle, min_a, max_a: angle < max_a - 15,
                'recommendation': "Straighten your leg more",
                'severity': 'medium'
            }
        },
        'movement_speed_check': True,
        'min_rep_duration': 2.0
    },

    'circular_pendulum_standing': {
        'joint': 'shoulder',
        'min_angle': 10,
        'max_angle': 60,
        'errors': {
            'excessive_motion': {
                'condition': lambda angle, min_a, max_a: angle > max_a + 10,
                'recommendation': "Make smaller circles with your arm",
                'severity': 'medium'
            },
            'insufficient_motion': {
                'condition': lambda angle, min_a, max_a: angle < min_a + 5,
                'recommendation': "Move your arm in a bigger circle",
                'severity': 'low'
            },
            'stopped_motion': {
                'condition': lambda angle, min_a, max_a: False,
                'recommendation': "Keep your arm moving gently",
                'severity': 'medium'
            }
        },
        'continuous_motion': True,
        'movement_speed_check': False
    },

    'external_rotation_shoulders_elastic': {
        'joint': 'elbow',
        'min_angle': 70,
        'max_angle': 110,
        'errors': {
            'elbow_too_open': {
                'condition': lambda angle, min_a, max_a: angle > max_a + 10,
                'recommendation': "Keep your elbow closer to your body",
                'severity': 'high'
            },
            'elbow_too_closed': {
                'condition': lambda angle, min_a, max_a: angle < min_a - 10,
                'recommendation': "Open your elbow slightly",
                'severity': 'medium'
            },
            'fast_motion': {
                'condition': lambda angle, min_a, max_a: False,
                'recommendation': "Control the elastic band slowly",
                'severity': 'medium'
            }
        },
        'movement_speed_check': True,
        'min_rep_duration': 2.5
    },

    'horizontal_weighted_openings_standing': {
        'joint': 'shoulder',
        'min_angle': 60,
        'max_angle': 120,
        'errors': {
            'insufficient_opening': {
                'condition': lambda angle, min_a, max_a: angle < min_a + 10,
                'recommendation': "Open your arms wider",
                'severity': 'medium'
            },
            'excessive_opening': {
                'condition': lambda angle, min_a, max_a: angle > max_a + 10,
                'recommendation': "Do not over-extend your arms",
                'severity': 'high'
            },
            'insufficient_closing': {
                'condition': lambda angle, min_a, max_a: angle > max_a - 10,
                'recommendation': "Bring your arms closer together",
                'severity': 'low'
            }
        },
        'movement_speed_check': True,
        'min_rep_duration': 2.0
    },

    'lift_extended_leg_supine': {
        'joint': 'hip',
        'min_angle': 30,
        'max_angle': 70,
        'secondary_joint': 'knee',
        'secondary_min': 160,
        'secondary_max': 180,
        'errors': {
            'leg_not_straight': {
                'condition': lambda angle, min_a, max_a: False,
                'recommendation': "Keep your leg straight",
                'severity': 'high'
            },
            'insufficient_lift': {
                'condition': lambda angle, min_a, max_a: angle < min_a + 10,
                'recommendation': "Lift your leg higher",
                'severity': 'medium'
            },
            'excessive_lift': {
                'condition': lambda angle, min_a, max_a: angle > max_a + 10,
                'recommendation': "Do not raise your leg too high",
                'severity': 'high'
            },
            'leg_lowered_incomplete': {
                'condition': lambda angle, min_a, max_a: angle > min_a - 5,
                'recommendation': "Lower your leg completely",
                'severity': 'low'
            }
        },
        'movement_speed_check': True,
        'min_rep_duration': 2.5
    },

    'shoulder_flexion_seated': {
        'joint': 'shoulder',
        'min_angle': 10,
        'max_angle': 170,
        'errors': {
            'insufficient_raise': {
                'condition': lambda angle, min_a, max_a: angle < max_a - 30,
                'recommendation': "Lift your arm higher",
                'severity': 'medium'
            },
            'excessive_raise': {
                'condition': lambda angle, min_a, max_a: angle > max_a + 15,
                'recommendation': "Lower your arm slightly",
                'severity': 'low'
            },
            'incomplete_lowering': {
                'condition': lambda angle, min_a, max_a: angle > min_a + 20,
                'recommendation': "Lower your arm completely",
                'severity': 'medium'
            }
        },
        'movement_speed_check': True,
        'min_rep_duration': 2.0
    }
}

## 4. Voice Feedback Module

Text-to-speech feedback with cooldown to prevent spam.

In [ ]:
class VoiceFeedback:

    def __init__(self, rate=150, volume=0.9, cooldown=3.0):
        self.engine = pyttsx3.init()
        self.engine.setProperty('rate', rate)
        self.engine.setProperty('volume', volume)
        self.enabled = True
        self.is_speaking = False
        self.last_messages = {}
        self.cooldown = cooldown
        self.lock = threading.Lock()

    def speak(self, text, force=False):
        if not self.enabled or not text:
            return False

        current_time = time.time()

        with self.lock:
            if not force and text in self.last_messages:
                if current_time - self.last_messages[text] < self.cooldown:
                    return False
            self.last_messages[text] = current_time

        thread = threading.Thread(target=self._speak_thread, args=(text,))
        thread.daemon = True
        thread.start()
        return True

    def _speak_thread(self, text):
        self.is_speaking = True
        try:
            self.engine.say(text)
            self.engine.runAndWait()
        except:
            pass
        finally:
            self.is_speaking = False

    def enable(self):
        self.enabled = True

    def disable(self):
        self.enabled = False

    def clear_cooldown(self):
        with self.lock:
            self.last_messages.clear()

    def stop(self):
        try:
            self.engine.stop()
        except:
            pass

## 5. Exercise Validator

Core validation logic for pose keypoints and joint angle analysis.

In [ ]:
class ExerciseState(Enum):
    """Exercise movement states for repetition counting"""
    IDLE = "idle"
    MIN_POSITION = "min"
    TRANSITION = "transition"
    MAX_POSITION = "max"
    INVALID = "invalid"


@dataclass
class JointAngleRule:
    """Defines angle validation rules for a joint"""
    keypoint_indices: Tuple[int, int, int]  # (point_a, joint_b, point_c)
    min_angle: float
    max_angle: float
    name: str


@dataclass
class ExerciseRule:
    """Complete rule definition for an exercise"""
    name: str
    primary_joint: JointAngleRule
    secondary_joint: Optional[JointAngleRule] = None
    sides: List[str] = None  # ['left', 'right'] or ['both']
    continuous_oscillation: bool = False


@dataclass
class ValidationResult:
    """Result of exercise validation"""
    is_valid: bool
    current_angle: float
    rep_count: int
    state: ExerciseState
    feedback: str
    side: str


# YOLOv8 Pose COCO Keypoint Indices
KEYPOINTS = {
    'nose': 0,
    'left_eye': 1,
    'right_eye': 2,
    'left_ear': 3,
    'right_ear': 4,
    'left_shoulder': 5,
    'right_shoulder': 6,
    'left_elbow': 7,
    'right_elbow': 8,
    'left_wrist': 9,
    'right_wrist': 10,
    'left_hip': 11,
    'right_hip': 12,
    'left_knee': 13,
    'right_knee': 14,
    'left_ankle': 15,
    'right_ankle': 16,
}


# Exercise Rules Dictionary for Validator
VALIDATOR_EXERCISE_RULES = {
    'bending_knee_no_support_seated': ExerciseRule(
        name='Bending knee no support seated',
        primary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_hip'], KEYPOINTS['left_knee'], KEYPOINTS['left_ankle']),
            min_angle=70,
            max_angle=180,
            name='knee'
        ),
        sides=['left', 'right']
    ),

    'bending_knee_bed_support_supine': ExerciseRule(
        name='Bending knee with bed support supine',
        primary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_hip'], KEYPOINTS['left_knee'], KEYPOINTS['left_ankle']),
            min_angle=60,
            max_angle=150,
            name='knee'
        ),
        sides=['left', 'right']
    ),

    'bending_knee_with_support_seated': ExerciseRule(
        name='Bending knee with support seated',
        primary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_hip'], KEYPOINTS['left_knee'], KEYPOINTS['left_ankle']),
            min_angle=80,
            max_angle=160,
            name='knee'
        ),
        sides=['left', 'right']
    ),

    'circular_pendulum_standing': ExerciseRule(
        name='Circular pendulum standing',
        primary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_hip'], KEYPOINTS['left_shoulder'], KEYPOINTS['left_elbow']),
            min_angle=10,
            max_angle=60,
            name='shoulder'
        ),
        sides=['left', 'right'],
        continuous_oscillation=True
    ),

    'external_rotation_shoulders_elastic': ExerciseRule(
        name='External rotation shoulders elastic',
        primary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_shoulder'], KEYPOINTS['left_elbow'], KEYPOINTS['left_wrist']),
            min_angle=70,
            max_angle=110,
            name='elbow'
        ),
        sides=['left', 'right']
    ),

    'horizontal_weighted_openings_standing': ExerciseRule(
        name='Horizontal weighted openings standing',
        primary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_elbow'], KEYPOINTS['left_shoulder'], KEYPOINTS['left_hip']),
            min_angle=60,
            max_angle=120,
            name='shoulder'
        ),
        sides=['left', 'right']
    ),

    'lift_extended_leg_supine': ExerciseRule(
        name='Lift extended leg supine',
        primary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_shoulder'], KEYPOINTS['left_hip'], KEYPOINTS['left_knee']),
            min_angle=30,
            max_angle=70,
            name='hip'
        ),
        secondary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_hip'], KEYPOINTS['left_knee'], KEYPOINTS['left_ankle']),
            min_angle=160,
            max_angle=180,
            name='knee_extension'
        ),
        sides=['left', 'right']
    ),

    'shoulder_flexion_seated': ExerciseRule(
        name='Shoulder flexion seated',
        primary_joint=JointAngleRule(
            keypoint_indices=(KEYPOINTS['left_hip'], KEYPOINTS['left_shoulder'], KEYPOINTS['left_elbow']),
            min_angle=10,
            max_angle=170,
            name='shoulder'
        ),
        sides=['left', 'right']
    ),
}

In [ ]:
class ExerciseValidator:
    """Real-time exercise validation using pose keypoints and angle analysis"""

    def __init__(self, exercise_key: str, confidence_threshold: float = 0.5):
        if exercise_key not in VALIDATOR_EXERCISE_RULES:
            raise ValueError(f"Unknown exercise: {exercise_key}. Available: {list(VALIDATOR_EXERCISE_RULES.keys())}")

        self.exercise = VALIDATOR_EXERCISE_RULES[exercise_key]
        self.confidence_threshold = confidence_threshold

        # State tracking for each side
        self.state = {'left': ExerciseState.IDLE, 'right': ExerciseState.IDLE}
        self.rep_count = {'left': 0, 'right': 0}
        self.angle_buffer = {'left': [], 'right': []}
        self.buffer_size = 3

        # Angle tolerance for state detection (degrees)
        self.min_tolerance = 15
        self.max_tolerance = 15

    def calculate_angle(self, point_a: np.ndarray, point_b: np.ndarray, point_c: np.ndarray) -> float:
        """Calculate angle at joint B formed by points A-B-C"""
        vector_ba = point_a - point_b
        vector_bc = point_c - point_b

        cos_angle = np.dot(vector_ba, vector_bc) / (
            np.linalg.norm(vector_ba) * np.linalg.norm(vector_bc) + 1e-6
        )
        cos_angle = np.clip(cos_angle, -1.0, 1.0)
        angle = np.degrees(np.arccos(cos_angle))

        return angle

    def check_keypoint_confidence(self, keypoints: np.ndarray, indices: Tuple[int, int, int]) -> bool:
        """Check if all required keypoints meet confidence threshold"""
        for idx in indices:
            if keypoints[idx, 2] < self.confidence_threshold:
                return False
        return True

    def get_side_adjusted_indices(self, side: str) -> Tuple[Tuple[int, int, int], Optional[Tuple[int, int, int]]]:
        """Get keypoint indices adjusted for left/right side"""
        primary = list(self.exercise.primary_joint.keypoint_indices)
        secondary = None

        if side == 'right':
            primary = [self._swap_side(idx) for idx in primary]
            if self.exercise.secondary_joint:
                secondary = [self._swap_side(idx) for idx in self.exercise.secondary_joint.keypoint_indices]
        else:
            if self.exercise.secondary_joint:
                secondary = list(self.exercise.secondary_joint.keypoint_indices)

        return tuple(primary), tuple(secondary) if secondary else None

    def _swap_side(self, keypoint_idx: int) -> int:
        """Swap left keypoint index to right equivalent"""
        keypoint_map = {
            1: 2, 2: 1,  # eyes
            3: 4, 4: 3,  # ears
            5: 6, 6: 5,  # shoulders
            7: 8, 8: 7,  # elbows
            9: 10, 10: 9,  # wrists
            11: 12, 12: 11,  # hips
            13: 14, 14: 13,  # knees
            15: 16, 16: 15,  # ankles
        }
        return keypoint_map.get(keypoint_idx, keypoint_idx)

    def validate_angle(self, angle: float) -> Tuple[bool, bool]:
        """Check if angle is within valid range and determine position"""
        min_angle = self.exercise.primary_joint.min_angle
        max_angle = self.exercise.primary_joint.max_angle

        is_valid = min_angle <= angle <= max_angle
        is_at_min = abs(angle - min_angle) <= self.min_tolerance
        is_at_max = abs(angle - max_angle) <= self.max_tolerance

        return is_valid, is_at_min, is_at_max

    def update_state_machine(self, angle: float, side: str) -> bool:
        """Update state machine for repetition counting"""
        is_valid, is_at_min, is_at_max = self.validate_angle(angle)

        if not is_valid:
            self.state[side] = ExerciseState.INVALID
            return False

        current_state = self.state[side]
        new_rep = False

        if self.exercise.continuous_oscillation:
            if is_at_min and current_state in [ExerciseState.MAX_POSITION, ExerciseState.TRANSITION]:
                self.state[side] = ExerciseState.MIN_POSITION
                self.rep_count[side] += 1
                new_rep = True
            elif is_at_max and current_state in [ExerciseState.MIN_POSITION, ExerciseState.TRANSITION]:
                self.state[side] = ExerciseState.MAX_POSITION
            elif not is_at_min and not is_at_max:
                self.state[side] = ExerciseState.TRANSITION
        else:
            if is_at_min:
                if current_state == ExerciseState.MAX_POSITION:
                    self.rep_count[side] += 1
                    new_rep = True
                self.state[side] = ExerciseState.MIN_POSITION
            elif is_at_max:
                if current_state == ExerciseState.MIN_POSITION:
                    self.state[side] = ExerciseState.MAX_POSITION
            elif is_valid:
                if current_state in [ExerciseState.MIN_POSITION, ExerciseState.MAX_POSITION]:
                    self.state[side] = ExerciseState.TRANSITION

        return new_rep

    def validate_frame(self, keypoints: np.ndarray, side: str = 'left') -> ValidationResult:
        """Validate single frame of pose keypoints"""
        primary_indices, secondary_indices = self.get_side_adjusted_indices(side)

        if not self.check_keypoint_confidence(keypoints, primary_indices):
            return ValidationResult(
                is_valid=False,
                current_angle=0.0,
                rep_count=self.rep_count[side],
                state=ExerciseState.INVALID,
                feedback=f"Low confidence on {side} {self.exercise.primary_joint.name}",
                side=side
            )

        point_a = keypoints[primary_indices[0], :2]
        point_b = keypoints[primary_indices[1], :2]
        point_c = keypoints[primary_indices[2], :2]

        primary_angle = self.calculate_angle(point_a, point_b, point_c)

        if self.exercise.secondary_joint and secondary_indices:
            if not self.check_keypoint_confidence(keypoints, secondary_indices):
                return ValidationResult(
                    is_valid=False,
                    current_angle=primary_angle,
                    rep_count=self.rep_count[side],
                    state=ExerciseState.INVALID,
                    feedback=f"Low confidence on {side} {self.exercise.secondary_joint.name}",
                    side=side
                )

            sec_point_a = keypoints[secondary_indices[0], :2]
            sec_point_b = keypoints[secondary_indices[1], :2]
            sec_point_c = keypoints[secondary_indices[2], :2]
            secondary_angle = self.calculate_angle(sec_point_a, sec_point_b, sec_point_c)

            sec_min = self.exercise.secondary_joint.min_angle
            sec_max = self.exercise.secondary_joint.max_angle
            if not (sec_min <= secondary_angle <= sec_max):
                return ValidationResult(
                    is_valid=False,
                    current_angle=primary_angle,
                    rep_count=self.rep_count[side],
                    state=ExerciseState.INVALID,
                    feedback=f"Keep {side} knee extended ({secondary_angle:.1f})",
                    side=side
                )

        self.angle_buffer[side].append(primary_angle)
        if len(self.angle_buffer[side]) > self.buffer_size:
            self.angle_buffer[side].pop(0)
        smoothed_angle = np.mean(self.angle_buffer[side])

        new_rep = self.update_state_machine(smoothed_angle, side)

        is_valid, is_at_min, is_at_max = self.validate_angle(smoothed_angle)

        if new_rep:
            feedback = f"Rep {self.rep_count[side]} complete!"
        elif not is_valid:
            if smoothed_angle < self.exercise.primary_joint.min_angle:
                feedback = f"Angle too small ({smoothed_angle:.1f})"
            else:
                feedback = f"Angle too large ({smoothed_angle:.1f})"
        elif is_at_min:
            feedback = "At minimum position"
        elif is_at_max:
            feedback = "At maximum position"
        else:
            feedback = "In transition"

        return ValidationResult(
            is_valid=is_valid,
            current_angle=smoothed_angle,
            rep_count=self.rep_count[side],
            state=self.state[side],
            feedback=feedback,
            side=side
        )

    def reset(self):
        """Reset all counters and state"""
        self.state = {'left': ExerciseState.IDLE, 'right': ExerciseState.IDLE}
        self.rep_count = {'left': 0, 'right': 0}
        self.angle_buffer = {'left': [], 'right': []}

In [ ]:
class MultiSideValidator:
    """Wrapper to validate both sides simultaneously"""

    def __init__(self, exercise_key: str, confidence_threshold: float = 0.5):
        self.left_validator = ExerciseValidator(exercise_key, confidence_threshold)
        self.right_validator = ExerciseValidator(exercise_key, confidence_threshold)
        self.exercise_key = exercise_key

    def validate_frame(self, keypoints: np.ndarray) -> Dict[str, ValidationResult]:
        """Validate both sides in a single frame"""
        results = {
            'left': self.left_validator.validate_frame(keypoints, 'left'),
            'right': self.right_validator.validate_frame(keypoints, 'right')
        }
        return results

    def get_total_reps(self) -> int:
        """Get combined rep count from both sides"""
        return self.left_validator.rep_count['left'] + self.right_validator.rep_count['right']

    def reset(self):
        """Reset both validators"""
        self.left_validator.reset()
        self.right_validator.reset()

## 6. Exercise Correction Module

Error detection and form correction feedback.

In [ ]:
class ExerciseCorrection:

    def __init__(self, voice_enabled=True, cooldown=3.0):
        self.voice = VoiceFeedback(cooldown=cooldown)
        if not voice_enabled:
            self.voice.disable()
        self.angle_history = deque(maxlen=30)
        self.rep_start_time = None
        self.last_state = None
        self.error_counts = {}
        self.motion_stopped_frames = 0

    def detect_errors(self, exercise_name, joint_angles, current_rep_state, is_valid_motion, side='left'):
        errors = []
        if exercise_name not in EXERCISE_RULES:
            return errors

        rules = EXERCISE_RULES[exercise_name]
        primary_joint = rules['joint']
        min_angle = rules['min_angle']
        max_angle = rules['max_angle']

        primary_angle = joint_angles.get(f"{side}_{primary_joint}")
        if primary_angle is None:
            return errors

        self.angle_history.append({
            'angle': primary_angle,
            'time': time.time(),
            'state': current_rep_state
        })

        for error_type, error_def in rules['errors'].items():
            if error_type == 'stopped_motion' and rules.get('continuous_motion'):
                if len(self.angle_history) >= 10:
                    recent_angles = [h['angle'] for h in list(self.angle_history)[-10:]]
                    angle_variance = max(recent_angles) - min(recent_angles)
                    if angle_variance < 5:
                        self.motion_stopped_frames += 1
                        if self.motion_stopped_frames > 15:
                            errors.append(ErrorDetection(
                                error_type='stopped_motion',
                                severity=error_def['severity'],
                                recommendation=error_def['recommendation'],
                                joint_name=primary_joint,
                                current_value=primary_angle,
                                expected_range=(min_angle, max_angle)
                            ))
                    else:
                        self.motion_stopped_frames = 0
                continue

            if error_type == 'fast_motion':
                continue

            if error_def['condition'](primary_angle, min_angle, max_angle):
                errors.append(ErrorDetection(
                    error_type=error_type,
                    severity=error_def['severity'],
                    recommendation=error_def['recommendation'],
                    joint_name=primary_joint,
                    current_value=primary_angle,
                    expected_range=(min_angle, max_angle)
                ))

        if 'secondary_joint' in rules:
            sec_joint = rules['secondary_joint']
            sec_angle = joint_angles.get(f"{side}_{sec_joint}")
            if sec_angle is not None:
                sec_min = rules['secondary_min']
                sec_max = rules['secondary_max']
                if not (sec_min <= sec_angle <= sec_max):
                    errors.append(ErrorDetection(
                        error_type='leg_not_straight',
                        severity='high',
                        recommendation="Keep your leg straight",
                        joint_name=sec_joint,
                        current_value=sec_angle,
                        expected_range=(sec_min, sec_max)
                    ))

        if rules.get('movement_speed_check'):
            if current_rep_state != self.last_state:
                if current_rep_state == 'min' and self.last_state == 'max':
                    if self.rep_start_time is not None:
                        rep_duration = time.time() - self.rep_start_time
                        min_duration = rules.get('min_rep_duration', 2.0)
                        if rep_duration < min_duration:
                            errors.append(ErrorDetection(
                                error_type='too_fast',
                                severity='medium',
                                recommendation="Slow down your movement",
                                joint_name=primary_joint,
                                current_value=rep_duration,
                                expected_range=(min_duration, 5.0)
                            ))
                    self.rep_start_time = time.time()
                elif current_rep_state == 'min':
                    self.rep_start_time = time.time()

        self.last_state = current_rep_state

        if not is_valid_motion and current_rep_state != 'neutral':
            errors.append(ErrorDetection(
                error_type='incomplete_rep',
                severity='medium',
                recommendation="Complete the full movement",
                joint_name=primary_joint,
                current_value=primary_angle,
                expected_range=(min_angle, max_angle)
            ))

        return errors

    def generate_recommendation(self, errors):
        if not errors:
            return None
        severity_priority = {'high': 3, 'medium': 2, 'low': 1}
        sorted_errors = sorted(errors, key=lambda e: severity_priority[e.severity], reverse=True)
        return sorted_errors[0].recommendation

    def provide_feedback(self, exercise_name, joint_angles, current_rep_state, is_valid_motion, side='left', force_speak=False):
        errors = self.detect_errors(exercise_name, joint_angles, current_rep_state, is_valid_motion, side)
        recommendation = self.generate_recommendation(errors)
        spoken = False
        if recommendation:
            spoken = self.voice.speak(recommendation, force=force_speak)
        return {
            'errors': errors,
            'recommendation': recommendation,
            'spoken': spoken,
            'error_count': len(errors),
            'highest_severity': errors[0].severity if errors else None
        }

    def enable_voice(self):
        self.voice.enable()

    def disable_voice(self):
        self.voice.disable()

    def reset(self):
        self.angle_history.clear()
        self.rep_start_time = None
        self.last_state = None
        self.error_counts.clear()
        self.motion_stopped_frames = 0
        self.voice.clear_cooldown()

    def get_statistics(self):
        return {
            'total_errors': sum(self.error_counts.values()),
            'error_breakdown': dict(self.error_counts),
            'angle_history_size': len(self.angle_history)
        }


def create_correction(voice_enabled=True, cooldown=3.0):
    return ExerciseCorrection(voice_enabled=voice_enabled, cooldown=cooldown)

## 7. Complete Exercise System

Main system combining pose detection, validation, and feedback.

In [ ]:
class CompleteSystem:

    def __init__(self, exercise_key):
        print(f"Loading system for: {exercise_key}")
        self.pose_model = YOLO('yolov8n-pose.pt')
        self.validator = MultiSideValidator(exercise_key, confidence_threshold=0.5)
        self.correction = create_correction(voice_enabled=True, cooldown=3.0)
        self.exercise_key = exercise_key

    def process_frame(self, frame):
        pose_results = self.pose_model(frame, verbose=False)
        if not pose_results or len(pose_results[0].keypoints.data) == 0:
            return None, frame

        keypoints = pose_results[0].keypoints.data[0].cpu().numpy()
        validation_results = self.validator.validate_frame(keypoints)

        left_result = validation_results['left']
        right_result = validation_results['right']
        active_side = 'left' if left_result.rep_count >= right_result.rep_count else 'right'
        active_result = validation_results[active_side]

        joint_angles = {
            f"{active_side}_knee": active_result.current_angle,
            f"{active_side}_shoulder": active_result.current_angle,
            f"{active_side}_hip": active_result.current_angle,
            f"{active_side}_elbow": active_result.current_angle,
        }

        state_mapping = {
            'IDLE': 'neutral',
            'MIN_POSITION': 'min',
            'MAX_POSITION': 'max',
            'TRANSITION': 'transition',
            'INVALID': 'neutral'
        }

        rep_state = state_mapping.get(active_result.state.value, 'neutral')

        feedback = self.correction.provide_feedback(
            exercise_name=self.exercise_key,
            joint_angles=joint_angles,
            current_rep_state=rep_state,
            is_valid_motion=active_result.is_valid,
            side=active_side
        )

        frame = self.draw_feedback(frame, validation_results, feedback)

        return {
            'validation': validation_results,
            'feedback': feedback,
            'total_reps': self.validator.get_total_reps(),
            'active_side': active_side
        }, frame

    def draw_feedback(self, frame, validation_results, feedback):
        h, w = frame.shape[:2]
        overlay = frame.copy()
        cv2.rectangle(overlay, (10, 10), (450, 300), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)

        y_pos = 40
        cv2.putText(frame, "LEFT SIDE", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        y_pos += 30

        left = validation_results['left']
        left_color = (0, 255, 0) if left.is_valid else (0, 0, 255)
        cv2.putText(frame, f"Angle: {left.current_angle:.1f} deg", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, left_color, 1)
        y_pos += 25
        cv2.putText(frame, f"Reps: {left.rep_count}", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        y_pos += 35

        cv2.putText(frame, "RIGHT SIDE", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        y_pos += 30

        right = validation_results['right']
        right_color = (0, 255, 0) if right.is_valid else (0, 0, 255)
        cv2.putText(frame, f"Angle: {right.current_angle:.1f} deg", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, right_color, 1)
        y_pos += 25
        cv2.putText(frame, f"Reps: {right.rep_count}", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        y_pos += 35

        if feedback['recommendation']:
            cv2.rectangle(frame, (15, y_pos - 20), (435, y_pos + 30), (0, 255, 255), 2)
            cv2.putText(frame, "TIP:", (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
            y_pos += 25
            cv2.putText(frame, feedback['recommendation'], (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

        total = self.validator.get_total_reps()
        cv2.putText(frame, f"TOTAL: {total}", (w - 200, h - 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
        cv2.putText(frame, "Q:Quit | R:Reset | V:Voice", (20, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        return frame

    def run(self):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("Error: Could not open webcam")
            return

        print("System running. Press Q=quit, R=reset, V=toggle voice\n")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            result, annotated_frame = self.process_frame(frame)

            if result is None:
                cv2.putText(annotated_frame, "NO PERSON DETECTED", (100, 360), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

            cv2.imshow('Complete Exercise System', annotated_frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('r'):
                self.validator.reset()
                self.correction.reset()
                print("System reset")
            elif key == ord('v'):
                if self.correction.voice.enabled:
                    self.correction.disable_voice()
                    print("Voice OFF")
                else:
                    self.correction.enable_voice()
                    print("Voice ON")

        cap.release()
        cv2.destroyAllWindows()

        print("\n" + "=" * 60)
        print("SESSION SUMMARY")
        print("=" * 60)
        print(f"Exercise: {self.exercise_key}")
        print(f"Total Reps: {self.validator.get_total_reps()}")
        print(f"Left: {self.validator.left_validator.rep_count['left']} reps")
        print(f"Right: {self.validator.right_validator.rep_count['right']} reps")
        print("=" * 60 + "\n")

## 8. List Available Exercises

In [ ]:
# Display available exercises
print("\n" + "=" * 60)
print("AVAILABLE EXERCISES")
print("=" * 60 + "\n")

for i, (key, rule) in enumerate(EXERCISE_RULES.items(), 1):
    print(f"{i}. {key}")
    print(f"   Joint: {rule['joint']}")
    print(f"   Angle Range: {rule['min_angle']}° - {rule['max_angle']}°")
    print()

## 9. Run the System

Select an exercise and start the webcam-based validation system.

In [ ]:
# Select exercise (change the index as needed)
exercise_list = list(EXERCISE_RULES.keys())

# Choose exercise by index (1-8)
selected_index = 1  # Change this to select different exercise
selected_exercise = exercise_list[selected_index - 1]

print(f"Selected Exercise: {selected_exercise}")
print("\nStarting system...")

In [ ]:
# Initialize and run the system
system = CompleteSystem(selected_exercise)
system.run()

## 10. Interactive Exercise Selection (Alternative)

Run this cell for interactive exercise selection using dropdown widget.

In [ ]:
# Interactive selection using ipywidgets (if available)
try:
    import ipywidgets as widgets
    from IPython.display import display
    
    exercise_dropdown = widgets.Dropdown(
        options=list(EXERCISE_RULES.keys()),
        value=list(EXERCISE_RULES.keys())[0],
        description='Exercise:',
        style={'description_width': 'initial'}
    )
    
    run_button = widgets.Button(description='Start Exercise')
    output = widgets.Output()
    
    def on_button_click(b):
        with output:
            output.clear_output()
            selected = exercise_dropdown.value
            print(f"Starting: {selected}")
            system = CompleteSystem(selected)
            system.run()
    
    run_button.on_click(on_button_click)
    
    display(exercise_dropdown, run_button, output)
    
except ImportError:
    print("ipywidgets not installed. Use manual selection in cell 9.")